# Convert Sinhala SQuAD to JSONL

This notebook converts the Hugging Face dataset to the same field format as `new_split_v2/test.jsonl`.

In [ ]:
%pip install -q datasets huggingface_hub

In [ ]:
from pathlib import Path
import json

from datasets import load_dataset
from huggingface_hub import login

DATASET_NAME = "Sachin-Hansaka/SQAD-Sinhala_Question_Answering_Dataset"
HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"
# This Hugging Face dataset contains only a 'train' split.
SPLIT = "train"
OUTPUT_PATH = Path("/tmp/sqad_test.jsonl")

In [ ]:
if HF_TOKEN == "hf_PASTE_YOUR_TOKEN_HERE":
    raise ValueError("Paste your Hugging Face token into HF_TOKEN first.")

login(token=HF_TOKEN, add_to_git_credential=False)
dataset = load_dataset(DATASET_NAME, split=SPLIT, token=HF_TOKEN)
dataset

In [ ]:
def first_answer(answers):
    if isinstance(answers, dict):
        texts = answers.get("text", [])
        return texts[0] if texts else ""
    if isinstance(answers, list) and answers:
        answer = answers[0]
        return answer.get("text", "") if isinstance(answer, dict) else str(answer)
    return ""


OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8", newline="\n") as output:
    for item in dataset:
        answer = first_answer(item.get("answers", []))
        record = {
            "question": item.get("question", ""),
            "answer": answer,
            "context": item.get("context", ""),
            "answerable": bool(answer),
            "grade": "",
            "chapter": "",
            "chapter_title": item.get("title", ""),
        }
        output.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(dataset):,} records to {OUTPUT_PATH.resolve()}")

In [ ]:
# Preview the first 20 and last 20 converted records.
from collections import deque

preview_count = 20
first_records = []
last_records = deque(maxlen=preview_count)

with OUTPUT_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file):
        if not line.strip():
            continue
        record = json.loads(line)
        if len(first_records) < preview_count:
            first_records.append(record)
        last_records.append(record)

print(f"START OF DATASET — FIRST {len(first_records)} RECORDS")
for number, record in enumerate(first_records, 1):
    print(f"\n--- Record {number} ---")
    print(json.dumps(record, ensure_ascii=False, indent=2))

print(f"\nEND OF DATASET — LAST {len(last_records)} RECORDS")
for number, record in enumerate(last_records, 1):
    print(f"\n--- Record {number} ---")
    print(json.dumps(record, ensure_ascii=False, indent=2))